## PDF Extraction and Embedding (Programme Documents)

### 1. Extracting raw text data

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader, UnstructuredPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import os, re

In [2]:
eee_path = "https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/"
pdf_path = "../RAG/ProgramBooklet"

path = os.path.join(pdf_path)
pdfs = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
llm = ChatOllama(model="deepseek-r1:8b", validate_model_on_init=True, temperature=0.6, reasoning=False)
print(pdfs)

['MSc_EIE_46011_2526.pdf', 'BEngBSc_Scheme_IAIE_46409_2526.pdf', 'PhDMPhil_EEE_46601_2526.pdf', 'MSc_EE_46010_2526.pdf', 'BEng_Scheme_EE_46408_2526.pdf', 'MSc_EV_46012_2526.pdf', 'MSc_MQ_46013_2526.pdf']


### Please install "poppler" and "tesseract" in Homebrew before proceeding to the next step!
Warning! Long running time! (Approx. 45 mins per documents)

In [ ]:
def regex_enhance(txt):
    text = re.sub(r' {2,}', ' ', txt)  # Strip excess white spaces
    return text

def extract_programme_title(first_page_content, llm_model):
    """
    Extract programme title from first page using LLM.
    """
    # Take first 2000 characters to avoid token limits
    content_snippet = first_page_content[:2000]
    
    prompt = \
        f"""
        You are extracting the title from a university programme booklet's first page.

        Extract the full programme title including:
            1. Degree type (e.g., Bachelor of Engineering, Master of Science, PhD)
            2. Discipline (e.g., Electrical Engineering, Electronic and Information Engineering)

        Return only the programme title with normalized capitalization, nothing else!

        *First page content*:
        {content_snippet}

        Programme Title:
        """
    
    # Extract programme title from first page using LLM
    try:
        response = llm_model.invoke(prompt)
        title = response.content.strip()
        
        # Clean up the response
        title = re.sub(r'\s+', ' ', title)
        title = title.replace('"', '').replace("'", "")
        
        # Validate it's a reasonable title
        if len(title) > 10 and len(title) < 250:
            return title
        else:
            return "Unknown"
    except Exception as e:
        print(f"LLM extraction failed: {e}")
        return "Unknown"

text_docs = []
table_docs = []
unwanted_metadata = ["producer", "creator", "creationdate", "file_path", 
                     "format", "title", "subject", "keywords", "moddate", 
                     "author", "trapped", "modDate", "creationDate"]

for pdf in pdfs:
    loader = PyMuPDFLoader(f"{pdf_path}/{pdf}")
    cur_pdf = loader.load()
    
    # Extract programme title from first page
    programme_title = ""
    if len(cur_pdf) > 0:
        programme_title = extract_programme_title(cur_pdf[0].page_content, llm)
        print(f"PDF: {pdf} -> Programme: {programme_title}")
    
    # Load with Unstructured for elements
    loader = UnstructuredPDFLoader(
        f"{pdf_path}/{pdf}", 
        mode="elements", 
        strategy="hi_res"   # Use hi_res strategy for better table extraction
    )
    elements = loader.load()
    
    for element in elements:
        # Augment metadata
        element.metadata["source"] = pdf
        element.metadata["content_type"] = "pdf"
        element.metadata["programme_title"] = programme_title
        for key in unwanted_metadata:
            if key in element.metadata:
                del element.metadata[key]
        
        if element.metadata.get("category") == "Table":
            # Summarize table
            prompt = f"Summarize the following table in natural language:\n\n{element.page_content}"
            summary = llm.invoke(prompt).content.strip()
            element.metadata["table_content"] = element.page_content  # store original table
            element.page_content = summary
            element.metadata["content_type"] = "table_summary"
            table_docs.append(element)
        else:
            text_docs.append(element)

In [ ]:
print(f"Number of text elements: {len(text_docs)}, table elements: {len(table_docs)}")
# print(text_docs[0].page_content if text_docs else "No text docs")

### 2. Text Splitting

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(text_docs)

all_chunks = chunks + table_docs

for i, chunk in enumerate(all_chunks):
    source = chunk.metadata.get("source", "N/A")
    page = chunk.metadata.get("page", "N/A")
    programme = chunk.metadata.get("programme_title", "N/A")
    chunk_type = "table" if chunk.metadata.get("content_type") == "table_summary" else "chunk"
    chunk.metadata["chunk_id"] = f"PolyU_Doc_{source}_page_{page}_{chunk_type}_{i}"
    del chunk.metadata["programme_title"]
    
    chunk.page_content = f"--- Source: {source}, Programme: {programme} --- \n --- Retrieved from: {eee_path} --- \n\n{chunk.page_content}"

In [ ]:
print(all_chunks[0])

page_content='--- Source: MSc_EIE_46011_2526.pdf, Programme: Master Of Science In Electronic And Information Engineering --- 
 --- Retrieved from: https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/ --- 

completion of the late assessment. 
 
The student concerned is required to submit his/her application for late assessment in writing to the 
Head of Department offering the subject, within five working days from the date of the examination, 
together with any original supporting documents. Approval of applications for late assessment 
and the means for such late assessments shall be given by the Head of Department offering the 
subject or the subject teacher concerned, in consultation with the Programme Leader. Verification 
of the supporting documents with the issuing authority may be conducted by the subject offering 
Department as part of the approval process. 
 
5.11 Assessment to be competed 
 
For cases where students fail marginally in one o

### 3. Document Embedding in Chroma

In [ ]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "academic_documents" if not SINGLE else "vaa_documents"

In [ ]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

141

In [ ]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, all_chunks))

for i, chunk in enumerate(all_chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i)]
    )

print(f"Added {len(all_chunks)} chunks into ChromaDB")

Added 3179 chunks into ChromaDB


### 4. Simple Testing

In [ ]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "What is Master of Science of Electronic and Information Engineering?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content[:]}...")
    print(f"Source: {result.metadata.get('source')}, Page: {result.metadata.get('page')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: --- Source: MSc_EE_46010_2526.pdf, Programme: Master Of Science In Electrical Engineering --- 
 --- Retrieved from: https://www.polyu.edu.hk/eee/study/information-for-current-students/programme-documents/ --- 

Master of Science in Electrical Engineering 2025/26 
 
1 
 
1 
General Information 
 
1.1 
Programme Information 
 
Programme Title (Code) 
 
Master of Science in Electrical Engineering (46010) 
電機工程學理學碩士學位 
 
Host Department 
 
Department of Electrical and Electronic Engineering 
 
Mode of Study and Normal Duration 
 
Mode 
Normal Duration 
Mixed-Mode 
Full-time: 1.5 years (3 semesters) 
Part-time: 2.5 years (5 semesters) 
 
Students should complete the programme within the normal duration of the programme. Those 
who exceed the normal duration of the programme will be de-registered from the programme 
unless prior approval has been obtained from relevant authorities. 
 
Award Title 
 
Students will be awarded one of the following awards upon successful completion of t

/var/folders/cf/1j9rc9v11rzf5wxcjw3w_wsm0000gp/T/ipykernel_82117/618414439.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(
